# 03 – PostgreSQL database

**Goal:** Store the integrated tables in PostgreSQL (container in the Codespace) and query them from Python with SQL, including joins and aggregations.
**Input:** `data/interim/*.parquet` (notebook 02) · **Output:** filled database tables
**Assessment items:** P3 (database with SQL queries from Python), B2 (PostgreSQL instead of SQLite, joins and aggregations)

Schema: `sql/schema.sql` – primary and foreign keys connect `circuits` → `races` → `results` / `laps` / `weather` / `pairs`, and `drivers` → `driver_map` / `results` / `laps` / `pairs`.

In [1]:
import sys
from pathlib import Path

# Make the repository root importable, whether Jupyter starts in the root or in notebooks/
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").is_dir())
sys.path.insert(0, str(ROOT))

import pandas as pd

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)

In [2]:
from sqlalchemy import text

from src.config import INTERIM_DIR
from src.db import TABLE_ORDER, get_engine, load_tables, reset_schema, table_counts

engine = get_engine()
with engine.connect() as conn:
    print(conn.execute(text("SELECT version()")).scalar_one().split(",")[0])

PostgreSQL 16.15 (Ubuntu 16.15-0ubuntu0.24.04.1) on x86_64-pc-linux-gnu


## 1. Create schema and load the tables

`reset_schema` drops and recreates all project tables, so the notebook always produces the same database state.

In [3]:
reset_schema(engine)
tables = {name: pd.read_parquet(INTERIM_DIR / f"{name}.parquet")
          for name in TABLE_ORDER if name != "pairs"}
load_tables(engine, tables)
pd.Series(table_counts(engine)).to_frame("rows")

,rows
circuits,26
driver_map,43
drivers,26
laps,47333
pairs,0
races,44
results,880
weather,7342


## 2. SQL queries from Python

**Query 1 – races and laps per era and circuit type** (join `races` × `circuits`, aggregation)

In [4]:
pd.read_sql("""
    SELECT r.era,
           c.circuit_type,
           COUNT(*)                 AS races,
           ROUND(AVG(r.total_laps)) AS avg_laps,
           ROUND(AVG(c.length_km), 3) AS avg_length_km
    FROM races r
    JOIN circuits c USING (circuit_id)
    GROUP BY r.era, c.circuit_type
    ORDER BY r.era, races DESC
""", engine)

,era,circuit_type,races,avg_laps,avg_length_km
0,E1,race,19,61.0,5.129
1,E1,street,3,60.0,5.171
2,E2,race,15,61.0,5.179
3,E2,street,7,61.0,5.072


**Query 2 – pit stops and neutralised laps per race** (join `laps` × `races` × `circuits`, conditional aggregation)

In [5]:
per_race = pd.read_sql("""
    SELECT r.season, r.round, r.race_name, c.circuit_type,
           COUNT(*)                                       AS laps,
           COUNT(*) FILTER (WHERE l.pit_in)               AS pit_stops,
           ROUND(100.0 * AVG((l.track_status ~ '[4567]')::int), 1) AS pct_laps_sc_vsc_red
    FROM laps l
    JOIN races r    USING (race_id)
    JOIN circuits c USING (circuit_id)
    GROUP BY r.season, r.round, r.race_name, c.circuit_type
    ORDER BY r.season, r.round
""", engine)
per_race.head(10)

,season,round,race_name,circuit_type,laps,pit_stops,pct_laps_sc_vsc_red
0,2021,1,Bahrain Grand Prix,race,1027,43,9.3
1,2021,2,Emilia Romagna Grand Prix,race,1127,57,14.9
2,2021,3,Portuguese Grand Prix,race,1245,60,7.7
3,2021,4,Spanish Grand Prix,race,1247,37,4.7
4,2021,5,Monaco Grand Prix,street,1419,20,0.0
5,2021,6,Azerbaijan Grand Prix,street,943,60,14.3
6,2021,7,French Grand Prix,race,1051,22,0.0
7,2021,8,Styrian Grand Prix,race,1296,26,0.0
8,2021,9,Austrian Grand Prix,race,1338,28,4.3
9,2021,10,British Grand Prix,race,970,42,4.0


In [6]:
per_race.groupby("season")[["pit_stops", "pct_laps_sc_vsc_red"]].mean().round(1)

,pit_stops,pct_laps_sc_vsc_red
season,,
2021,38.5,8.1
2022,38.0,10.2


**Query 3 – positions gained from grid to finish per team and season** (join `results` × `races`, aggregation)

In [7]:
pd.read_sql("""
    SELECT r.season, res.team,
           COUNT(*) AS starts,
           ROUND(AVG(res.grid - res.finish_pos), 2) AS avg_positions_gained
    FROM results res
    JOIN races r USING (race_id)
    WHERE res.grid > 0 AND res.finish_pos IS NOT NULL
    GROUP BY r.season, res.team
    ORDER BY r.season, avg_positions_gained DESC
""", engine)

,season,team,starts,avg_positions_gained
0,2021,Alfa Romeo,41,1.10
1,2021,Haas F1 Team,44,0.64
2,2021,Aston Martin,43,0.49
3,2021,McLaren,44,0.48
4,2021,Alpine F1 Team,44,0.34
5,2021,Ferrari,44,0.07
6,2021,Williams,43,-0.26
7,2021,Mercedes,44,-1.07
8,2021,Red Bull,40,-1.23
9,2021,AlphaTauri,41,-1.24
